# 02 Preprocessing
Tokenization of the texts, the vocabulary, the coverage of the pretrained word embeddings, and the embedding matrix that both models use.

Load the train, validation and test splits (like on slide 9 of the course project slides).

In [15]:
from datasets import load_dataset

train = load_dataset("amarfurt/art-nlp-hs26", split="train")
valid = load_dataset("amarfurt/art-nlp-hs26", split="validation")
test = load_dataset("amarfurt/art-nlp-hs26", split="test")

print(len(train), len(valid), len(test))

16801 1000 1532


## Tokenization
NLTK's tokenization needs the `punkt_tab` resource, a pretrained model that `word_tokenize` uses to split text into sentences. It is downloaded first.


In [28]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mmici\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

Compare splitting at whitespace with NLTK's `word_tokenize` on one example (the first short training hypothesis that contains both a comma and a contraction).


In [17]:
text = train[269]['hypothesis_1']
print(text)
print(text.split())
print(nltk.word_tokenize(text))

He didn't listen to them, and had to go to the hospital.
['He', "didn't", 'listen', 'to', 'them,', 'and', 'had', 'to', 'go', 'to', 'the', 'hospital.']
['He', 'did', "n't", 'listen', 'to', 'them', ',', 'and', 'had', 'to', 'go', 'to', 'the', 'hospital', '.']


`split()` cuts the text only at spaces, so punctuation stays attached to words (`them,`, `hospital.`). `word_tokenize` separates punctuation (`them` and `,`) and contractions (`did` and `n't`) from the words. This way the words match the vocabulary of the word embeddings, and negations like `n't` stay their own tokens. `word_tokenize` is used for all texts.

## Word embeddings
Load the pretrained GloVe embeddings (300 dimensions) with gensim's downloader, and look at the vector size, the vocabulary size and the first words.

In [18]:
import gensim.downloader as api
wv = api.load('glove-wiki-gigaword-300')


In [19]:
print(f'Vector size: {wv.vector_size}')
print(f'Vocabulary size: {len(wv.index_to_key)}')
for index, word in enumerate(wv.index_to_key[:10]):
    print(f"word #{index}: {word}")

Vector size: 300
Vocabulary size: 400000
word #0: the
word #1: ,
word #2: .
word #3: of
word #4: to
word #5: and
word #6: in
word #7: a
word #8: "
word #9: 's


Count the words in the training data. Tokenize all texts (O1, O2, h1, h2) and count how many words there are in total (tokens) and how many distinct words (types), once as written and once lowercased. This gives the list of words that need a vector.

In [20]:
columns = ['observation_1', 'observation_2', 'hypothesis_1', 'hypothesis_2']

tokens = []
for column in columns:
    for text in train[column]:
        tokens.extend(nltk.word_tokenize(text))

fdist = nltk.FreqDist(tokens)
fdist_lower = nltk.FreqDist(token.lower() for token in tokens)
print('Tokens:', len(tokens))
print('Types:', len(fdist))
print('Types (lowercased):', len(fdist_lower))
print(f'Type-token-ratio: {len(fdist) / len(tokens):.2%}')

Tokens: 655477
Types: 17789
Types (lowercased): 16210
Type-token-ratio: 2.71%


Share of types and tokens that have a vector in GloVe, and the most frequent words without a vector.


In [21]:
def coverage(wv, fdist):
    known = [word for word in fdist if word in wv]
    known_tokens = sum(fdist[word] for word in known)
    print(f'Known types: {len(known) / len(fdist):.2%}')
    print(f'Known tokens: {known_tokens / fdist.N():.2%}')
    unknown = [word for word, count in fdist.most_common() if word not in wv]
    print('Most frequent unknown:', unknown[:15])

print('GloVe, cased:')
coverage(wv, fdist)
print('GloVe, lowercased:')
coverage(wv, fdist_lower)


GloVe, cased:
Known types: 73.82%
Known tokens: 87.43%
Most frequent unknown: ['I', 'The', 'He', 'She', 'My', 'It', 'A', 'They', 'Gina', 'Amy', 'Bob', 'Tim', 'Sam', 'We', 'John']
GloVe, lowercased:
Known types: 95.40%
Known tokens: 99.84%
Most frequent unknown: ['mturk', '..', 'selfie', 'zaiden', 'a+', 'antonela', 'fariah', 'myah', 'shalil', 'rubix', 'kelsea', 'jacelyn', 'lamay', 'marisse', 'lakyn']


GloVe only contains lowercase words: cased, only 87.43% of the tokens have a vector; lowercased, 99.84%, so almost every word the model sees has a vector. The remaining unknown words are rare names, typos and *mturk*. **All texts are therefore lowercased.**

For comparison: the coverage of fastText and word2vec (the embeddings from the course exercise). Each is loaded, measured and deleted again to free memory.

In [22]:
for name in ['fasttext-wiki-news-subwords-300', 'word2vec-google-news-300']:
    other_wv = api.load(name)
    print(f'{name}, cased:')
    coverage(other_wv, fdist)
    print(f'{name}, lowercased:')
    coverage(other_wv, fdist_lower)
    del other_wv


fasttext-wiki-news-subwords-300, cased:
Known types: 96.95%
Known tokens: 99.38%
Most frequent unknown: ["n't", '``', "''", 'mTurk', 'Zaiden', 'mturk', 'A+', 'Antonela', 'Fariah', 'Mturk', 'Shalil', 'rubix', 'Marisse', 'Lakyn', 'Tanice']
fasttext-wiki-news-subwords-300, lowercased:
Known types: 93.97%
Known tokens: 99.11%
Most frequent unknown: ["n't", '``', "''", 'kelsi', 'marcy', 'rhonda', 'joanie', 'mindy', 'mturk', 'horace', 'tabitha', 'hallie', 'horatio', 'juanita', 'callie']
word2vec-google-news-300, cased:
Known types: 94.12%
Known tokens: 78.34%
Most frequent unknown: ['.', 'to', 'a', 'and', ',', 'of', "'s", '!', '``', "'", '10', '20', 'didnt', '100', 'cancelled']
word2vec-google-news-300, lowercased:
Known types: 88.86%
Known tokens: 77.62%
Most frequent unknown: ['.', 'to', 'a', 'and', ',', 'of', "'s", '!', '``', "'", '10', '20', 'didnt', 'tami', 'maddie']


Conclusion: GloVe covers 99.84% of the training tokens. word2vec, used in the course exercise, covers only about 78%: it has no vectors for punctuation and frequent words like *to*, *a*, *and*, *of*. fastText covers 99.38%, but has no vector for `n't`, so all negations (*didn't*, *wasn't*) would be lost. So GloVe is more suitable for this project.


## Stopwords 
Download NLTK's English stopword list and check which important words it contains.

In [23]:
nltk.download('stopwords')
stopwords = set(nltk.corpus.stopwords.words('english'))
print('Stopwords:', len(stopwords))
print([word for word in ['not', 'no', 'never', 'before', 'after', 'then'] if word in stopwords])

Stopwords: 198
['not', 'no', 'before', 'after', 'then']


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mmici\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


How many words in the texts are stopwords? Share of the training tokens that are stopwords.

In [24]:
stopword_tokens = sum(fdist_lower[word] for word in fdist_lower if word in stopwords)
print(f'Stopword tokens: {stopword_tokens / fdist_lower.N():.2%}')

Stopword tokens: 38.95%


Number of hypothesis pairs in which h1 and h2 differ only in stopwords. 

In [25]:
only_stopwords = 0
for h1, h2 in zip(train['hypothesis_1'], train['hypothesis_2']):
    words_1 = set(word.lower() for word in nltk.word_tokenize(h1))
    words_2 = set(word.lower() for word in nltk.word_tokenize(h2))
    difference = words_1 ^ words_2  # words that occur in only one of the two hypotheses
    if difference and difference <= stopwords:
        only_stopwords += 1 
print('Pairs that differ only in stopwords:', only_stopwords)
print(f'Share: {only_stopwords / len(train):.2%}')

Pairs that differ only in stopwords: 367
Share: 2.18%


NLTK's stopword list contains *not*, *no*, *before*, *after* and *then*, words that change the meaning of a sentence. Stopwords make up 38.95% of the training tokens, and in 367 pairs (2.18%) the two hypotheses differ only in stopwords; after removing them, h1 and h2 would be identical. **Stopwords are therefore kept.**

## Preprocessing function
All decisions above in one function: tokenize with NLTK, lowercase all words, keep stopwords.

In [26]:
def preprocess(text):
    # tokenize
    text = nltk.word_tokenize(text)
    # lowercase all words
    text = [word.lower() for word in text]
    # remove stopwords (not used, see above)
    # text = [word for word in text if word not in stopwords]
    return text

print(train[269]["hypothesis_1"])
print(preprocess(train[269]["hypothesis_1"]))

He didn't listen to them, and had to go to the hospital.
['he', 'did', "n't", 'listen', 'to', 'them', ',', 'and', 'had', 'to', 'go', 'to', 'the', 'hospital', '.']


In [27]:
print(preprocess("The Cat didn't sleep."))


['the', 'cat', 'did', "n't", 'sleep', '.']


## Sentence vectors
Each text becomes one vector: the average of the GloVe vectors of its words (course slides "Word Embeddings", slide 32). Words without a GloVe vector are skipped.

In [31]:
words = preprocess(train[269]['hypothesis_1'])
vectors = [wv[word] for word in words if word in wv]
print(len(words), len(vectors))
print(vectors[0].shape)

15 15
(300,)


Average the word vectors: one vector with 300 numbers for the whole sentence.

In [36]:
import numpy as np

sentence_vector = np.mean(vectors, axis=0)
print(sentence_vector.shape)
print(sentence_vector[:5])

(300,)
[-0.21954928  0.03110227 -0.24757503 -0.11346499 -0.11204454]


The same steps as a function that works for any text. A text without any known word in GloVe gets a zero vector.

In [35]:
def average_vector(text):
    words = preprocess(text)
    vectors = [wv[word] for word in words if word in wv]
    # no known word: zero vector
    if len(vectors) == 0:
        return np.zeros(wv.vector_size)
    return np.mean(vectors, axis=0)

print(average_vector(train[269]["hypothesis_1"])[:5])
print(average_vector("mturk").shape)
print(average_vector("mturk")[:5])

[-0.21954928  0.03110227 -0.24757503 -0.11346499 -0.11204454]
(300,)
[0. 0. 0. 0. 0.]


## Input format
One example becomes one input: the sentence vectors of O1, O2, h1 and h2 side by side (4 x 300 = 1200 numbers). The label (1 or 2) becomes the class 0 or 1.

In [37]:
columns = ["observation_1", "observation_2", "hypothesis_1", "hypothesis_2"]

example = train[0]
x = np.concatenate([average_vector(example[column]) for column in columns])
y = example["label"] - 1
print(x.shape)
print(example["label"], y)

(1200,)
2 1


Apply this to all examples of each split: one row of 1200 numbers per example, and one class per example.

In [38]:
def example_vector(example):
    return np.concatenate([average_vector(example[column]) for column in columns])

X_train = np.array([example_vector(example) for example in train])
y_train = np.array([example["label"] - 1 for example in train])
print(X_train.shape, y_train.shape)

(16801, 1200) (16801,)


## Sources
- Dataset: course version of ART, `amarfurt/art-nlp-hs26` on Hugging Face (A. Marfurt), based on Bhagavatula et al. (2020), *Abductive Commonsense Reasoning*, ICLR 2020, https://arxiv.org/abs/1908.05739.
- Loading the splits: code from slide 9 of the course slides "Course Project HS26" (A. Marfurt).
- Tokenization: `nltk.download('punkt_tab')` and `nltk.word_tokenize` as in the course exercise "NLP Pipeline" (Week 1).
- Tokens/types and `nltk.FreqDist` as in the course exercise "NLP Pipeline" (Week 1); coverage check (`word in wv`).
- Word embeddings: GloVe (Pennington et al., 2014), fastText (Bojanowski et al., 2017) and word2vec (Mikolov et al., 2013), loaded with gensim's downloader (Řehůřek & Sojka, 2010) as in the course exercise "Word Embeddings" (Week 2).
- Stopwords: NLTK's English stopword list (`nltk.corpus.stopwords`) as in the course exercise "Topic Modeling" (Week 2).
- Preprocessing function: structure of `preprocess()` as in the course exercise "Topic Modeling" (Week 2).
- Tools: code written with the help of Claude (Anthropic); all code reviewed and understood by me.
